# Staged XAS correlator at a probe time
Plot the raw real-time XAS correlator and compare a Gaussian two-time transform with a one-sided relative-time transform. Run this notebook from its directory. Choose the two XAS CSVs produced by `my_hubbard_chain_td_XAS_staged.py`; the singletime input can be the combined CSV or the step CSV matching the selected t′.

Both XAS transforms use the same greater-correlator sign: exp(−iω(t−t′)) for the double-time transform and exp(−iωτ) for the one-sided transform. Their windows, prefactors and normalizations differ. They do not automatically include a core-level energy, lifetime broadening, a spin sum, or a finite-duration X-ray probe model. The one-sided quantity shown is a Gaussian-windowed proxy, not automatically a complete experimental XAS intensity.

In [ ]:
from pathlib import Path
import csv
import sys
import numpy as np
import matplotlib.pyplot as plt

code_dir = Path.cwd() / 'code'  # Or use the full path to code/
if not (code_dir / 'plotting_td_dos.py').is_file():
    raise FileNotFoundError(f'Code directory not found: {code_dir.resolve()}')
sys.path.insert(0, str(code_dir.resolve()))
from plotting_td_dos import fixed_probe_response

two_xas = Path('/Users/qqt/Documents/Projects/r_rtcvsttc/code/2026_11_hubbard_chain_localandtwotime_forwardandbackward_cpu_v2.1/N=4_Nup=2_Ndown=2_20261119_001324/processed_data/XAS>_singletime_td_N=4_Nup=2_Ndown=2_20261119_013727_057672_freq_0/step_0015')
single_xas = Path('/Users/qqt/Documents/Projects/r_rtcvsttc/code/2026_11_hubbard_chain_localandtwotime_forwardandbackward_cpu_v2.1/N=4_Nup=2_Ndown=2_20261119_001324/processed_data/XAS>_singletime_td_N=4_Nup=2_Ndown=2_20261119_013727_057672_freq_0/step_0015.csv')  # Combined or matching step CSV
site_pair = (1, 1)
probe_time = 8.0  # Must exist as t_prime in the singletime file
sigma = 3.0
relative_sigma = sigma * np.sqrt(2)
omega_min, omega_max, omega_count = -15.0, 15.0, 401
frequency_shift = 0.0  # Display frequency = transform frequency - shift
output_csv = Path('td_xas_probe_response.csv')


## Calculate the response
The singletime `t_prime` must match `probe_time` within 1e−9; no interpolation or nearest-time substitution is used. The output CSV is overwritten.

In [ ]:
for path in (two_xas, single_xas):
    if not path.is_file():
        raise FileNotFoundError(f'Input not found: {path.resolve()}')
    if path.resolve() == output_csv.resolve():
        raise ValueError('Output CSV must not overwrite an input')
if omega_count < 2 or omega_min >= omega_max:
    raise ValueError('Use at least two frequencies and omega_min < omega_max')
frequencies = np.linspace(omega_min, omega_max, omega_count)
result = fixed_probe_response({'greater': two_xas}, {'greater': single_xas}, site_pair, probe_time, frequencies, sigma, relative_sigma, {'greater': frequency_shift}, relative_signs={'greater': -1})
low, high = result['nyquist_display_bounds']
display_min, display_max = max(omega_min, low), min(omega_max, high)
if display_min >= display_max:
    raise ValueError('Requested frequency grid does not intersect the common Nyquist window')
print(f'Common shifted Nyquist window: [{low:g}, {high:g}]')
with output_csv.open('w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(('center_site', 'site_index', 'probe_time', 'omega_shifted', 'two_time_real', 'two_time_imag', 'relative_response_real', 'relative_response_imag', 'relative_intensity'))
    for w, two, relative, intensity in zip(frequencies, result['two_time'], result['relative_response'], result['relative_intensity']):
        writer.writerow((*site_pair, probe_time, w, two.real, two.imag, relative.real, relative.imag, intensity))
print(f'Wrote {output_csv}')


## Compare transforms
The one-sided intensity is −Im(response)/π with a Gaussian relative-time window. Limiting the plot to the Nyquist window does not remove aliasing already folded into the sampled data.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True, constrained_layout=True)
axes[0].plot(frequencies, result['two_time'].real, label='Real')
axes[0].plot(frequencies, result['two_time'].imag, label='Imaginary')
axes[0].set(title='XAS two-time transform', ylabel='Gaussian transform')
axes[1].plot(frequencies, result['relative_intensity'], label='Intensity')
axes[1].set(title='XAS singletime transform', ylabel='−Im(response) / π')
for ax in axes:
    ax.set(xlabel='Shifted frequency', xlim=(display_min, display_max))
    ax.legend()
    ax.grid(alpha=0.3)
fig.suptitle(f'Sites {site_pair}; probe time {probe_time:g}')
plt.show()


## Raw singletime correlator
Inspect the real and imaginary parts before Fourier transformation.

In [ ]:
data = result['time_slices']['greater']
fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
ax.plot(data['tau'], data['correlator'].real, label='Real')
ax.plot(data['tau'], data['correlator'].imag, label='Imaginary')
ax.set(xlabel='Relative time τ', ylabel='Raw correlator', title=f'XAS at t′ = {probe_time:g}; sites {site_pair}')
ax.legend()
ax.grid(alpha=0.3)
plt.show()
